# LastSeen — Baseline Demo

**Goal:** Turn room/desk photos into a simple searchable visual memory.

This baseline:
1. Uploads one or more photos.
2. Uses a pretrained YOLO object detector.
3. Shows detected objects with bounding boxes.
4. Stores object name, confidence, image name, and bounding box.
5. Lets you search for an object and see the images where it was detected.

No model training is required for this first baseline.


## 1. Load the pretrained detector
We use a small pretrained YOLO model so the demo runs quickly in Colab.


In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image
import pandas as pd
import os
from datetime import datetime

model = YOLO("yolo11n.pt")
print("Model loaded!")


## 2. Upload room/desk photos
Upload a few `.jpg`, `.jpeg`, or `.png` images.

For a useful demo, try photos containing common objects such as a laptop, phone, bottle, backpack, chair, or cup.


In [ ]:
uploaded = files.upload()

image_files = [
    filename for filename in uploaded.keys()
    if filename.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Uploaded {len(image_files)} image(s):")
for filename in image_files:
    print(" -", filename)


## 3. Detect objects and build the visual memory
Each detected object is stored together with the image in which it appeared.


In [ ]:
visual_memory = []

for image_name in image_files:
    results = model(image_name, verbose=False)
    result = results[0]

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        object_name = model.names[class_id]
        x1, y1, x2, y2 = [float(x) for x in box.xyxy[0]]

        visual_memory.append({
            "object": object_name,
            "confidence": confidence,
            "image": image_name,
            "bbox": [x1, y1, x2, y2],
            "scanned_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    # Save image with YOLO bounding boxes
    result.save(filename=f"detected_{image_name}")

print(f"Stored {len(visual_memory)} detections.")


## 4. View detection results
This is useful for the Final Project Sharing Session because you can directly show the early object-detection results.


In [ ]:
for image_name in image_files:
    output_name = f"detected_{image_name}"
    print(f"\nResults for {image_name}:")
    display(Image(filename=output_name))


## 5. Inspect the searchable visual memory


In [ ]:
memory_df = pd.DataFrame(visual_memory)

if len(memory_df) == 0:
    print("No objects were detected.")
else:
    display(memory_df[["object", "confidence", "image", "scanned_at"]].sort_values(
        "confidence", ascending=False
    ))


## 6. Search for a misplaced item
Try searches such as:

```python
search_item("cell phone")
search_item("laptop")
search_item("bottle")
```

The baseline returns images containing that detected object.


In [ ]:
def search_item(item_name):
    matches = [
        item for item in visual_memory
        if item["object"].lower() == item_name.lower()
    ]

    if not matches:
        print(f'No previous detection found for "{item_name}".')
        print("\nObjects currently in memory:")
        print(sorted(set(item["object"] for item in visual_memory)))
        return []

    # Most confident detections first
    matches = sorted(matches, key=lambda x: x["confidence"], reverse=True)

    print(f'Found {len(matches)} detection(s) for "{item_name}":')

    shown_images = set()

    for match in matches:
        print(
            f'- {match["image"]} | '
            f'confidence: {match["confidence"]:.2f} | '
            f'scanned: {match["scanned_at"]}'
        )

        if match["image"] not in shown_images:
            display(Image(filename=f'detected_{match["image"]}'))
            shown_images.add(match["image"])

    return matches


In [ ]:
# Change this to an object detected in your photos.
matches = search_item("cell phone")


## 7. Optional: Search using partial text
This makes the demo slightly friendlier. For example, searching `phone` can match `cell phone`.


In [ ]:
def search_item_flexible(query):
    query = query.lower().strip()

    matches = [
        item for item in visual_memory
        if query in item["object"].lower()
    ]

    if not matches:
        print(f'No previous detection found for "{query}".')
        print("\nTry one of these detected objects:")
        print(sorted(set(item["object"] for item in visual_memory)))
        return []

    matches = sorted(matches, key=lambda x: x["confidence"], reverse=True)

    print(f'Found {len(matches)} detection(s) matching "{query}":')
    shown_images = set()

    for match in matches:
        print(
            f'- {match["object"]} in {match["image"]} '
            f'(confidence: {match["confidence"]:.2f})'
        )

        if match["image"] not in shown_images:
            display(Image(filename=f'detected_{match["image"]}'))
            shown_images.add(match["image"])

    return matches


In [ ]:
# Example:
search_item_flexible("phone")


## What this baseline demonstrates

**Current pipeline**

`Room/desk photo → pretrained object detection → store detections → search object → retrieve relevant image`

### Good points to mention during the 3-minute sharing session
- This is an end-to-end baseline; no custom training is used yet.
- It tests whether pretrained object detection is sufficient for common misplaced items.
- The baseline also reveals limitations: small or unusual objects (for example keys, glasses, or specific earbuds) may be missed or may not belong to the detector's predefined classes.
- Those failures give us clear directions for later project iterations, such as open-vocabulary detection, improved small-object detection, richer natural-language search, and better "last seen" tracking across scans.
